In [ ]:
import os
import sys

!{sys.executable} --version
if sys.version_info.minor == 8:
    raise RuntimeError('USE JUPYTER KERNEL VENV 3.10/310/DEFAULT INSTEAD')

!cd /workspace/CRYPTO_MACAQUES && pip install .
!cd /home/crypto/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && source venv/bin/activate && !{sys.executable} -m pip install .

from IPython.core.display_functions import clear_output
clear_output()

In [ ]:
%load_ext autoreload
%autoreload

from datetime import timedelta, datetime
from dateutil import parser

from SRC.LIBRARIES.new_data_utils import fetch
from SRC.CORE._CONSTANTS import _KIEV_TIMESTAMP
import SRC.LIBRARIES.new_utils as nu


SYM = 'eth'.upper()
SYMBOL = f'{SYM}USDT'
TF = '15' + 'm'.upper()
CAPITAL_PER_TRADE = 1000
COMMISSION_RATE = 0.00075
SLIPPAGE = 0.0002
FORCE_CLOSE_AT_END = False

# ========== КРАСНЫЙ КАНАЛ ==========
ENABLE_RED_CHANNEL = True
RED_MIN_BREAK_BODY_TO_ATR = 0#.797
RED_ENABLE_DIRECTION_CHANGE_FILTER = True
RED_DIRECTION_CHANGE_LOOKBACK = 20#19s
RED_TP_LEVEL = 'yellow'
RED_TP_MODE = 'fixed'
RED_SL_RATIO = 0.8
RED_REQUIRE_GREEN_TOUCH_AFTER_SL = True
RED_MIN_BARS_OUTSIDE = 1
RED_MAX_BARS_OUTSIDE = 2
RED_STOCH_FILTER = True
RED_STOCH_SERIES = 'D'
RED_STOCH_OVERSOLD = 30
RED_STOCH_OVERBOUGHT = 70
RED_ENABLE_SIGNED_OI_SLOPE_FILTER = True
RED_SIGNED_OI_SLOPE_THRESHOLD = 0.0
RED_OI_REGRESSION_LOOKBACK_5M = 6

# ========== ЗЕЛЁНЫЙ КАНАЛ ==========
ENABLE_GREEN_CHANNEL = False
# TODO 0.5?
GREEN_MIN_BREAK_BODY_TO_ATR = 0.505
GREEN_TP_LEVEL = 'green'
GREEN_TP_MODE = 'fixed'
GREEN_REQUIRE_YELLOW_TOUCH_AFTER_SL = True
GREEN_MIN_BARS_OUTSIDE = 1
GREEN_MAX_BARS_OUTSIDE = 1000
GREEN_STOCH_FILTER = False
GREEN_STOCH_SERIES = 'K'
GREEN_STOCH_OVERSOLD = 30
GREEN_STOCH_OVERBOUGHT = 70

# TODO Futures? Сравнить результаты обоих вариантов.
market_type = 'SPOT'
mrc_buffer = 1100
stoch_buffer = 50
atr_buffer = 200

In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple
from SRC.LIBRARIES.binance_metrics.constants import DATA_DIR, BINANCE_ZIP_TF_MINUTES

# ========== ОРИГИНАЛЬНЫЕ ФУНКЦИИ ==========
def load_from_cache_or_fetch_df(tf: str, cache_subdir: str, start_dt: datetime, end_dt: datetime) -> pd.DataFrame:
    cache_dir = DATA_DIR / cache_subdir / SYMBOL
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / f"{tf}_{start_dt}_{end_dt}_{market_type}.parquet"

    if cache_path.exists():
        print(f"Loading {tf} from cache: {cache_path}")

        return pd.read_parquet(cache_path)

    print(f"{tf} cache not found. Downloading...")
    fetched_df = fetch(market_type, SYMBOL, tf, start_dt, end_dt)
    fetched_df.to_parquet(cache_path)
    print(f"{tf} cache saved: {cache_path}")

    return fetched_df


def calculate_meanline_direction_changes(df_counter: pd.DataFrame, lookback: int) -> pd.Series:
    meanline_diff = df_counter['meanline'].diff()
    slope_sign = np.sign(meanline_diff)
    direction_changes = pd.Series(index=df_counter.index, dtype=float)

    for i in range(lookback, len(df_counter)):
        signs = slope_sign.iloc[i-lookback+1:i+1]
        non_zero_signs = signs[signs != 0]

        if len(non_zero_signs) <= 1:
            direction_changes.iloc[i] = 0
        else:
            changes = (non_zero_signs != non_zero_signs.shift(1)).sum() - 1
            direction_changes.iloc[i] = changes

    return direction_changes


def calculate_linear_regression_metrics(
    series: pd.Series,
    lookback: int,
) -> pd.DataFrame:
    if lookback < 2:
        raise RuntimeError("lookback must be at least 2")

    values = series.to_numpy(dtype=float, copy=False)

    slopes = np.full(len(values), np.nan)
    r2 = np.full(len(values), np.nan)

    x = np.arange(lookback, dtype=float)
    x_mean = x.mean()
    x_centered = x - x_mean

    denominator = np.sum(x_centered ** 2)

    for i in range(lookback - 1, len(values)):
        y = values[i - lookback + 1:i + 1]

        if np.isnan(y).any():
            continue

        y_mean = y.mean()
        y_centered = y - y_mean

        numerator = np.sum(x_centered * y_centered)

        slope = numerator / denominator
        slopes[i] = slope

        y_pred = y_mean + slope * x_centered

        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y_mean) ** 2)

        if ss_tot == 0:
            r2[i] = 1.0
        else:
            r2[i] = 1.0 - ss_res / ss_tot

    return pd.DataFrame(
        {
            "slope": slopes,
            "r2": r2,
        },
        index=series.index,
    )


def add_regression_metrics(df: pd.DataFrame, source_column: str, lookback: int) -> pd.DataFrame:
    metrics = calculate_linear_regression_metrics(df[source_column], lookback)

    df[f"{source_column}_slope"] = metrics["slope"]
    df[f"{source_column}_abs_slope"] = metrics["slope"].abs()
    df[f"{source_column}_r2"] = metrics["r2"]

    return df


def get_df_back(df):
    return df.copy()


def get_enter(df_back, df_counter, band, condition_series, min_bars_outside, max_bars_outside):
    candle_bodies_outside_band_counter = 0
    len_condition_series = len(condition_series)
    candle_bodies_outside_band = np.zeros(len_condition_series)

    for i in range(len_condition_series):
        if condition_series.iloc[i]:
            candle_bodies_outside_band_counter += 1
        else:
            candle_bodies_outside_band_counter = 0

        candle_bodies_outside_band[i] = candle_bodies_outside_band_counter

    count_series = pd.Series(candle_bodies_outside_band, index=condition_series.index)
    previous_candle_bodies_outside_band_count = np.roll(count_series, 1)
    previous_candle_bodies_outside_band_count = previous_candle_bodies_outside_band_count[len(df_counter) - len(df_back):]
    candle_touches_band = ((df_back['low'] <= df_back[band]) & (df_back['high'] >= df_back[band])).values

    return candle_touches_band & (previous_candle_bodies_outside_band_count >= min_bars_outside) & (previous_candle_bodies_outside_band_count <= max_bars_outside)


def get_candle_body_part_below_loband(df_counter, loband):
    return (df_counter['open'] < df_counter[loband]) | (df_counter['close'] < df_counter[loband])


def get_candle_body_part_above_upband(df_counter, upband):
    return (df_counter['open'] > df_counter[upband]) | (df_counter['close'] > df_counter[upband])


def get_long_enter(df_back, df_counter, loband, min_bars_outside, max_bars_outside):
    return pd.Series(
        get_enter(df_back, df_counter, loband, get_candle_body_part_below_loband(df_counter, loband), min_bars_outside, max_bars_outside),
        index=df_back.index
    )


def get_short_enter(df_back, df_counter, upband, min_bars_outside, max_bars_outside):
    return pd.Series(
        get_enter(df_back, df_counter, upband, get_candle_body_part_above_upband(df_counter, upband), min_bars_outside, max_bars_outside),
        index=df_back.index
    )


def select_stoch_values(df_back: pd.DataFrame, stoch_series: str) -> pd.Series:
    return df_back['stoch_k' if stoch_series.upper() == 'K' else 'stoch_d']


def check_min_bars_outside(min_bars_outside: int):
    if min_bars_outside < 1:
        raise Exception('min_bars_outside < 1')


def get_db_back_range(df_back):
    return range(0, len(df_back))


def get_current_candle_data(df_back, i):
    candle = df_back.iloc[i]

    return candle['high'], candle['low'], candle['loband1'], candle['upband1'], candle['loband2'], candle['upband2'], candle['meanline']


def get_df_counter_index(current_i, find_break_candle_idx_offset):
    return current_i + find_break_candle_idx_offset


def find_break_candle_idx(df_counter, break_idx_band, min_bars_outside, find_break_candle_idx_offset, current_i):
    previous_i = get_df_counter_index(current_i, find_break_candle_idx_offset) - 1

    # Определяем условие для тела свечи за каналом
    if 'loband' in break_idx_band:
        condition = get_candle_body_part_below_loband(df_counter, break_idx_band)
    else:
        condition = get_candle_body_part_above_upband(df_counter, break_idx_band)

    counter = 0
    # Ищем начало серии
    for j in range(previous_i, -1, -1):
        if condition.iloc[j]:
            counter += 1
        else:
            break

    # Первая свеча в серии
    first_break_idx = previous_i - counter + 1

    # Проверяем, что серия достаточно длинная
    if counter >= min_bars_outside:
        return first_break_idx

    return -1


def get_break_candle_data(df_counter, break_idx_band, min_bars_outside, find_break_candle_idx_offset, i):
    """Возвращает параметры свечи пробоя"""
    break_idx = find_break_candle_idx(df_counter, break_idx_band, min_bars_outside, find_break_candle_idx_offset, i)

    if break_idx < 0:
        return None

    candle = df_counter.iloc[break_idx]
    body_size = abs(candle['close'] - candle['open'])
    atr_value = candle['atr']

    return {
        'break_idx': break_idx,
        'body_size': body_size,
        'range_size': candle['high'] - candle['low'],
        'body_to_atr': body_size / atr_value if atr_value > 0 else 0
    }


def get_is_long_signal(long_enter, i):
    return long_enter.iloc[i]


def get_is_short_signal(short_enter, i):
    return short_enter.iloc[i]


def get_price_slipped_up(price, slippage):
    return price * (1 + slippage)


def get_price_slipped_down(price, slippage):
    return price * (1 - slippage)


def get_can_enter_by_slippage(df_back, entry_price, i):
    candle = df_back.iloc[i]

    return get_candle_touches_price(candle['low'], entry_price, candle['high'])


def get_can_enter_by_break_candle(can_enter, break_candle_data, min_break_body_to_atr):
    if break_candle_data and break_candle_data['body_to_atr'] < min_break_body_to_atr:
        can_enter = False

    return can_enter


def get_can_enter_by_stoch_filter(can_enter, stoch_filter, direction, stoch_values, stoch_oversold, stoch_overbought, i):
    if can_enter and stoch_filter:
        current_stoch = stoch_values.iloc[i]

        if (direction == 'long' and not (current_stoch <= stoch_oversold)) or (direction == 'short' and not (current_stoch >= stoch_overbought)):
            can_enter = False

    return can_enter


def get_can_enter_by_signed_oi_slope_filter(df_counter, df_1m, can_enter, entry_price, enable_signed_oi_slope_filter, direction, signed_oi_slope_threshold, find_break_candle_idx_offset, i):
    if not can_enter or direction == "short" or not enable_signed_oi_slope_filter:
        return can_enter

    tf_number = nu.get_tf_number(TF)
    tf_symbol = nu.get_tf_symbol(TF)
    #todo adapt for other TFs
    minutes_per_candle = tf_number

    if tf_symbol != "M" or minutes_per_candle < BINANCE_ZIP_TF_MINUTES:
        raise RuntimeError(f"get_can_enter_by_signed_oi_slope_filter does not support {TF} TF")

    df_counter_index = get_df_counter_index(i, find_break_candle_idx_offset)
    candle_open_datetime = df_counter.index[df_counter_index]
    entry_datetime = find_entry_datetime(tf=TF, df_1m=df_1m, candle_open_datetime=candle_open_datetime, entry_price=entry_price)
    entry_offset_minutes = int((entry_datetime - candle_open_datetime).total_seconds() // 60)

    if entry_offset_minutes >= minutes_per_candle:
        raise RuntimeError(f"Unexpected entry_offset_minutes={entry_offset_minutes}")

    entry_snapshot = f"m{(entry_offset_minutes // BINANCE_ZIP_TF_MINUTES) * BINANCE_ZIP_TF_MINUTES}"
    signed_oi_slope = df_counter.iloc[df_counter_index][f'sum_open_interest_pct_change_slope_{entry_snapshot}']

    if direction == "long":
        return signed_oi_slope > signed_oi_slope_threshold

    raise RuntimeError(f"Unknown direction: {direction}")


def get_last_trade_exit_data(trades):
    last_trade = trades[-1]

    return last_trade['exit_idx'], last_trade['exit_type']


def get_candle_touches_price(low, price, high):
    return low <= price <= high


def get_enter_amount(commission, capital_per_trade, entry_price):
    return (capital_per_trade * (1 - commission)) / entry_price


def get_position(direction, channel, i, entry_price, amount, tp_price, sl_price, tp_line, tp_mode, break_candle_data):
    pos = {
        'type': direction,
        'channel': channel,
        'entry_idx': i,
        'entry_price': entry_price,
        'amount': amount,
        'tp': tp_price,
        'sl': sl_price,
        'tp_line': tp_line,
        'tp_mode': tp_mode,
        'tp_distance': abs(entry_price - tp_price)
    }

    if break_candle_data:
        pos['break_body_to_atr'] = break_candle_data['body_to_atr']
        pos['break_idx'] = break_candle_data['break_idx']
    else:
        pos['break_body_to_atr'] = 0
        pos['break_idx'] = -1

    return pos


def calculate_pnl(position: Dict, exit_price: float, slippage: float, commission: float, capital_per_trade: float) -> Tuple[float, float]:
    direction = position['type']
    amount = position['amount']

    if direction == 'long':
        exit_price_slipped = get_price_slipped_down(exit_price, slippage)
        exit_proceeds = amount * exit_price_slipped * (1 - commission)
        return exit_proceeds - capital_per_trade, exit_price_slipped
    elif direction == 'short':
        exit_price_slipped = get_price_slipped_up(exit_price, slippage)
        entry_proceeds = amount * position['entry_price']
        exit_cost = amount * exit_price_slipped * (1 + commission)
        return entry_proceeds - exit_cost, exit_price_slipped


def get_trade(position, exit_idx, exit_price_slipped, pnl, exit_type):
    return {
        'channel': position['channel'],
        'type': position['type'],
        'entry_idx': position['entry_idx'],
        'exit_idx': exit_idx,
        'entry_price': position['entry_price'],
        'exit_price': exit_price_slipped,
        'pnl': pnl,
        'exit_type': exit_type,
        'break_body_to_atr': position.get('break_body_to_atr', 0)
    }


def exit_from_position(position, current_upband1, current_meanline, current_loband1, current_high, current_low, slippage, commission, capital_per_trade, trades, i):
    if position is not None:
        if position['tp_mode'] == 'dynamic':
            if position['tp_line'] == 'upband1':
                position['tp'] = current_upband1
            elif position['tp_line'] == 'meanline':
                position['tp'] = current_meanline
            elif position['tp_line'] == 'loband1':
                position['tp'] = current_loband1

        exit_price = None
        exit_type = None

        for col in ['tp', 'sl']:
            price = position[col]

            if get_candle_touches_price(current_low, price, current_high):
                exit_price = price
                exit_type = col

        if exit_price is not None:
            pnl, exit_price_slipped = calculate_pnl(position, exit_price, slippage, commission, capital_per_trade)
            trades.append(get_trade(position, i, exit_price_slipped, pnl, exit_type))
            position = None

    return position, trades


def force_close_position(position: Dict, df_back: pd.DataFrame, slippage: float, commission: float, capital_per_trade: float) -> Dict:
    exit_price = df_back.iloc[-1]['close']
    pnl, exit_price_slipped = calculate_pnl(position, exit_price, slippage, commission, capital_per_trade)

    return get_trade(position, len(df_back) - 1, exit_price_slipped, pnl, 'forced')


def get_metrics(total_pnl, win_rate, num_trades, profit_factor):
    return {
        'total_pnl': total_pnl,
        'win_rate': win_rate,
        'num_trades': num_trades,
        'profit_factor': profit_factor
    }


def calculate_metrics(trades: List[Dict]) -> Dict:
    if len(trades) == 0:
        return get_metrics(0, 0, 0, 0)

    trades_df = pd.DataFrame(trades)
    total_pnl = trades_df['pnl'].sum()
    win_rate = (trades_df['pnl'] > 0).mean()
    num_trades = len(trades_df)

    sum_profit = trades_df[trades_df['pnl'] > 0]['pnl'].sum()
    sum_loss = abs(trades_df[trades_df['pnl'] < 0]['pnl'].sum())
    profit_factor = sum_profit / sum_loss if sum_loss > 0 else float('inf')

    return get_metrics(total_pnl, win_rate, num_trades, profit_factor)


def find_entry_datetime(tf: str, df_1m: pd.DataFrame, candle_open_datetime: pd.Timestamp, entry_price: float) -> pd.Timestamp:
    tf_number = nu.get_tf_number(tf)
    tf_symbol = nu.get_tf_symbol(tf)
    #todo adapt for other TFs
    minutes_per_candle = tf_number

    if tf_symbol != "M" or minutes_per_candle < BINANCE_ZIP_TF_MINUTES:
        raise RuntimeError(f"find_entry_datetime does not support {tf} TF")

    df_1m_range = df_1m.loc[candle_open_datetime:candle_open_datetime + pd.Timedelta(minutes=minutes_per_candle - 1)]

    if len(df_1m_range) != minutes_per_candle:
        raise RuntimeError(f"Can`t retrieve one-minute candles from {tf} TF, got {len(df_1m_range)} for {candle_open_datetime}")

    for minute_time, candle in df_1m_range.iterrows():
        if get_candle_touches_price(candle["low"], entry_price, candle["high"]):
            return minute_time

    raise RuntimeError(f"Entry price {entry_price} not found in 1M candles for {candle_open_datetime}")


def backtest_red_channel(
    df: pd.DataFrame,
    df_counter: pd.DataFrame,
    df_1m: pd.DataFrame,
    commission: float = 0.00075,
    capital_per_trade: float = 1000.0,
    slippage: float = 0.0002,
    tp_level: str = 'far_green',
    tp_mode: str = 'fixed',
    sl_ratio: float = 3.0,
    require_green_touch_after_sl: bool = False,
    min_bars_outside: int = 0,
    max_bars_outside: int = 0,
    stoch_filter: bool = False,
    stoch_series: str = 'D',
    stoch_oversold: int = 20,
    stoch_overbought: int = 80,
    enable_signed_oi_slope_filter: bool = False,
    signed_oi_slope_threshold: float = 0.0,
    force_close_at_end: bool = False,
    min_break_body_to_atr: float = 0.0,
    find_break_candle_idx_offset: int = 0,
    enable_direction_change_filter: bool = False
) -> List[Dict]:
    df_back = get_df_back(df)
    long_enter = get_long_enter(df_back, df_counter, 'loband2', min_bars_outside, max_bars_outside)
    short_enter = get_short_enter(df_back, df_counter, 'upband2', min_bars_outside, max_bars_outside)
    stoch_values = select_stoch_values(df_back, stoch_series)
    position = None
    trades = []

    check_min_bars_outside(min_bars_outside)

    for i in get_db_back_range(df_back):
        current_high, current_low, current_loband1, current_upband1, current_loband2, current_upband2, current_meanline = get_current_candle_data(df_back, i)

        is_long_signal = get_is_long_signal(long_enter, i)
        is_short_signal = get_is_short_signal(short_enter, i)

        if position is None:
            direction = None
            entry_price = None
            tp_price = None
            tp_line = None

            if is_long_signal:
                direction = 'long'
                entry_price = get_price_slipped_up(current_loband2, slippage)
                break_idx_band = 'loband2'

                if tp_level == 'near_green':
                    tp_price = current_loband1
                    tp_line = 'loband1'
                elif tp_level == 'yellow':
                    tp_price = current_meanline
                    tp_line = 'meanline'
                elif tp_level == 'far_green':
                    tp_price = current_upband1
                    tp_line = 'upband1'

            elif is_short_signal:
                direction = 'short'
                entry_price = get_price_slipped_down(current_upband2, slippage)
                break_idx_band = 'upband2'

                if tp_level == 'near_green':
                    tp_price = current_upband1
                    tp_line = 'upband1'
                elif tp_level == 'yellow':
                    tp_price = current_meanline
                    tp_line = 'meanline'
                elif tp_level == 'far_green':
                    tp_price = current_loband1
                    tp_line = 'loband1'

            if direction is not None:
                break_candle_data = get_break_candle_data(df_counter, break_idx_band, min_bars_outside, find_break_candle_idx_offset, i)
                can_enter = get_can_enter_by_slippage(df_back, entry_price, i)
                can_enter = get_can_enter_by_break_candle(can_enter, break_candle_data, min_break_body_to_atr)
                can_enter = get_can_enter_by_stoch_filter(can_enter, stoch_filter, direction, stoch_values, stoch_oversold, stoch_overbought, i)
                can_enter = get_can_enter_by_signed_oi_slope_filter(df_counter, df_1m, can_enter, entry_price, enable_signed_oi_slope_filter, direction, signed_oi_slope_threshold, find_break_candle_idx_offset, i)

                if can_enter and require_green_touch_after_sl and len(trades) > 0:
                    last_exit_idx, last_exit_type = get_last_trade_exit_data(trades)

                    if last_exit_type == 'sl':
                        touched_green = False

                        for j in range(last_exit_idx + 1, i + 1):
                            low = df_back.iloc[j]['low']
                            high = df_back.iloc[j]['high']
                            loband1_j = df_back.iloc[j]['loband1']
                            upband1_j = df_back.iloc[j]['upband1']

                            if get_candle_touches_price(low, loband1_j, high) or get_candle_touches_price(low, upband1_j, high):
                                touched_green = True
                                break

                        if not touched_green:
                            can_enter = False

                if can_enter and enable_direction_change_filter:
                    direction_changes = df_counter.iloc[get_df_counter_index(i, find_break_candle_idx_offset)]['meanline_direction_changes']

                    if pd.isna(direction_changes) or direction_changes != 1:
                        can_enter = False

                if can_enter:
                    amount = get_enter_amount(commission, capital_per_trade, entry_price)

                    if direction == 'long':
                        distance_to_tp = tp_price - entry_price
                        sl_price = entry_price - distance_to_tp / sl_ratio
                    elif direction == 'short':
                        distance_to_tp = entry_price - tp_price
                        sl_price = entry_price + distance_to_tp / sl_ratio

                    position = get_position(direction, 'red', i, entry_price, amount, tp_price, sl_price, tp_line, tp_mode, break_candle_data)

                    continue

        position, trades = exit_from_position(position, current_upband1, current_meanline, current_loband1, current_high, current_low, slippage, commission, capital_per_trade, trades, i)

    # ========== ПРИНУДИТЕЛЬНОЕ ЗАКРЫТИЕ ==========
    if position is not None and force_close_at_end:
        trades.append(force_close_position(position, df_back, slippage, commission, capital_per_trade))

    return trades


def backtest_green_channel(
    df: pd.DataFrame,
    df_counter: pd.DataFrame,
    commission: float = 0.00075,
    capital_per_trade: float = 1000.0,
    slippage: float = 0.0002,
    tp_level: str = 'green',
    tp_mode: str = 'dynamic',
    require_yellow_touch_after_sl: bool = False,
    min_bars_outside: int = 0,
    max_bars_outside: int = 0,
    stoch_filter: bool = False,
    stoch_series: str = 'D',
    stoch_oversold: int = 20,
    stoch_overbought: int = 80,
    force_close_at_end: bool = False,
    min_break_body_to_atr: float = 0.0,
    find_break_candle_idx_offset: int = 0
) -> List[Dict]:
    df_back = get_df_back(df)
    long_enter = get_long_enter(df_back, df_counter, 'loband1', min_bars_outside, max_bars_outside)
    short_enter = get_short_enter(df_back, df_counter, 'upband1', min_bars_outside, max_bars_outside)
    stoch_values = select_stoch_values(df_back, stoch_series)
    position = None
    trades = []

    check_min_bars_outside(min_bars_outside)

    for i in get_db_back_range(df_back):
        current_high, current_low, current_loband1, current_upband1, current_loband2, current_upband2, current_meanline = get_current_candle_data(df_back, i)

        is_long_signal = get_is_long_signal(long_enter, i)
        is_short_signal = get_is_short_signal(short_enter, i)

        if position is None:
            direction = None
            entry_price = None
            tp_price = None
            tp_line = None

            if is_long_signal:
                direction = 'long'
                entry_price = get_price_slipped_up(current_loband1, slippage)
                break_idx_band = 'loband1'

                if tp_level == 'green':
                    tp_price = current_upband1
                    tp_line = 'upband1'
                elif tp_level == 'yellow':
                    tp_price = current_meanline
                    tp_line = 'meanline'

            elif is_short_signal:
                direction = 'short'
                entry_price = get_price_slipped_down(current_upband1, slippage)
                break_idx_band = 'upband1'

                if tp_level == 'green':
                    tp_price = current_loband1
                    tp_line = 'loband1'
                elif tp_level == 'yellow':
                    tp_price = current_meanline
                    tp_line = 'meanline'

            if direction is not None:
                break_candle_data = get_break_candle_data(df_counter, break_idx_band, min_bars_outside, find_break_candle_idx_offset, i)
                can_enter = get_can_enter_by_slippage(df_back, entry_price, i)
                can_enter = get_can_enter_by_break_candle(can_enter, break_candle_data, min_break_body_to_atr)
                can_enter = get_can_enter_by_stoch_filter(can_enter, stoch_filter, direction, stoch_values, stoch_oversold, stoch_overbought, i)

                if can_enter and require_yellow_touch_after_sl and len(trades) > 0:
                    last_exit_idx, last_exit_type = get_last_trade_exit_data(trades)

                    if last_exit_type == 'sl':
                        touched_yellow = False

                        for j in range(last_exit_idx + 1, i + 1):
                            low = df_back.iloc[j]['low']
                            high = df_back.iloc[j]['high']
                            meanline_j = df_back.iloc[j]['meanline']

                            if get_candle_touches_price(low, meanline_j, high):
                                touched_yellow = True
                                break

                        if not touched_yellow:
                            can_enter = False

                if can_enter:
                    amount = get_enter_amount(commission, capital_per_trade, entry_price)

                    if direction == 'long':
                        sl_price = current_loband2
                    elif direction == 'short':
                        sl_price = current_upband2

                    position = get_position(direction, 'green', i, entry_price, amount, tp_price, sl_price, tp_line, tp_mode, break_candle_data)

                    continue

        position, trades = exit_from_position(position, current_upband1, current_meanline, current_loband1, current_high, current_low, slippage, commission, capital_per_trade, trades, i)

    # ========== ПРИНУДИТЕЛЬНОЕ ЗАКРЫТИЕ ==========
    if position is not None and force_close_at_end:
        trades.append(force_close_position(position, df_back, slippage, commission, capital_per_trade))

    return trades

In [ ]:
# ========== ПЕРИОДЫ ДЛЯ ТЕСТИРОВАНИЯ ==========
# periods = [
#     ("2019-1-1 00:00", "2019-7-1 00:00"),
#     ("2019-7-1 00:00", "2020-1-1 00:00"),
#     ("2020-1-1 00:00", "2020-7-1 00:00"),
#     ("2020-7-1 00:00", "2021-1-1 00:00"),
#     ("2021-1-1 00:00", "2021-7-1 00:00"),
#     ("2021-7-1 00:00", "2022-1-1 00:00"),
#     ("2022-1-1 00:00", "2022-7-1 00:00"),
#     ("2022-7-1 00:00", "2023-1-1 00:00")
# ]

periods = [
    ("2021-12-2 00:00", "2022-1-1 00:00"),
    ("2022-1-1 00:00", "2022-7-1 00:00"),
    ("2022-7-1 00:00", "2023-1-1 00:00"),
    ("2023-1-1 00:00", "2023-7-1 00:00"),
    ("2023-7-1 00:00", "2024-1-1 00:00"),
    ("2024-1-1 00:00", "2024-7-1 00:00"),
    ("2024-7-1 00:00", "2025-1-1 00:00"),
    ("2025-1-1 00:00", "2025-7-1 00:00"),
    ("2025-7-1 00:00", "2026-1-1 00:00"),
    ("2026-1-1 00:00", "2026-6-1 00:00"),
    ("2026-6-1 00:00", "2026-6-30 00:00")
]

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import SRC.LIBRARIES.new_indicator_plot_utils as nipu

def show_fig_main():
    # ========== НАСТРОЙКИ ОТОБРАЖЕНИЯ ==========
    is_log_scale_by_default = False
    candlestick_row = 1

    # ========== СОЗДАНИЕ ГРАФИКА ==========
    fig_main = make_subplots(rows=1, cols=1, shared_xaxes=True, vertical_spacing=0.03)

    fig_main.add_trace(
        go.Candlestick(
            x=df[_KIEV_TIMESTAMP],
            open=df["open"],
            high=df["high"],
            low=df["low"],
            close=df["close"],
            name="OHLC"
        ),
        row=candlestick_row, col=1
    )

    nipu.add_mrc(candlestick_row, fig_main, df)

    # ========== НАСТРОЙКИ ГРАФИКА ==========
    price_log_scale_value = "log"
    price_linear_scale_value = "linear"
    price_log_scale_title = "Price (log scale)"
    price_linear_scale_title = "Price (linear scale)"

    fig_main.update_layout(
        title=f"🐵 {TF} | TP={RED_TP_LEVEL}({RED_TP_MODE}) | SL=1:{RED_SL_RATIO}",
        xaxis_rangeslider_visible=False,
        yaxis1_type=price_log_scale_value if is_log_scale_by_default else price_linear_scale_value,
        yaxis1_title=price_log_scale_title if is_log_scale_by_default else price_linear_scale_title,
        height=1200,
        bargap=0,
        updatemenus=[
            dict(
                type="buttons",
                direction="right",
                active=1 if is_log_scale_by_default else 0,
                x=0.315,
                y=1.073,
                buttons=[
                    dict(
                        label="Linear",
                        method="relayout",
                        args=[{"yaxis.type": price_linear_scale_value, "yaxis.title.text": price_linear_scale_title}]
                    ),
                    dict(
                        label="Log",
                        method="relayout",
                        args=[{"yaxis.type": price_log_scale_value, "yaxis.title.text": price_log_scale_title}]
                    )
                ],
                font=dict(color="red", size=12, family="Arial"),
                bgcolor="rgba(0, 0, 0, 0)",
                bordercolor="red",
                borderwidth=1
            )
        ]
    )

    # ========== МАРКЕРЫ СДЕЛОК ==========

    # Функция для получения цвета обводки в зависимости от стратегии
    def get_border_color(strategy):
        return 'darkred' if strategy == 'red' else 'darkgreen'

    # Собираем данные для маркеров
    trades_df = pd.DataFrame(all_trades) if all_trades else pd.DataFrame()

    if len(trades_df) > 0:
        # Long Entry
        long_entries = trades_df[trades_df['type'] == 'long']
        for type, group in long_entries.groupby('type'):
            border_color = get_border_color(type)
            fig_main.add_trace(
                go.Scatter(
                    x=df[_KIEV_TIMESTAMP].iloc[group['entry_idx']],
                    y=group['entry_price'],
                    mode='markers',
                    name=f'Long Entry ({type})',
                    marker=dict(
                        symbol='triangle-up',
                        size=10,
                        color='green',
                        line=dict(width=2, color=border_color)
                    )
                ),
                row=candlestick_row, col=1
            )

        # Short Entry
        short_entries = trades_df[trades_df['type'] == 'short']
        for type, group in short_entries.groupby('type'):
            border_color = get_border_color(type)
            fig_main.add_trace(
                go.Scatter(
                    x=df[_KIEV_TIMESTAMP].iloc[group['entry_idx']],
                    y=group['entry_price'],
                    mode='markers',
                    name=f'Short Entry ({type})',
                    marker=dict(
                        symbol='triangle-down',
                        size=10,
                        color='red',
                        line=dict(width=2, color=border_color)
                    )
                ),
                row=candlestick_row, col=1
            )

        # Take Profit (без trailing)
        tp_exits = trades_df[trades_df['exit_type'] == 'tp']
        for type, group in tp_exits.groupby('type'):
            border_color = get_border_color(type)
            fig_main.add_trace(
                go.Scatter(
                    x=df[_KIEV_TIMESTAMP].iloc[group['exit_idx']],
                    y=group['exit_price'],
                    mode='markers',
                    name=f'Take Profit ({type})',
                    marker=dict(
                        symbol='circle',
                        size=10,
                        color='lime',
                        opacity=0.7,
                        line=dict(width=2, color=border_color)
                    )
                ),
                row=candlestick_row, col=1
            )

        # TP (Trailing)
        tp_trailing_exits = trades_df[trades_df['exit_type'] == 'tp_trailing']
        for type, group in tp_trailing_exits.groupby('type'):
            border_color = get_border_color(type)
            fig_main.add_trace(
                go.Scatter(
                    x=df[_KIEV_TIMESTAMP].iloc[group['exit_idx']],
                    y=group['exit_price'],
                    mode='markers',
                    name=f'TP (Trailing) ({type})',
                    marker=dict(
                        symbol='diamond',
                        size=12,
                        color='lightgreen',
                        opacity=0.8,
                        line=dict(width=2, color=border_color)
                    )
                ),
                row=candlestick_row, col=1
            )

        # Stop Loss
        sl_exits = trades_df[trades_df['exit_type'] == 'sl']
        for type, group in sl_exits.groupby('type'):
            border_color = get_border_color(type)
            fig_main.add_trace(
                go.Scatter(
                    x=df[_KIEV_TIMESTAMP].iloc[group['exit_idx']],
                    y=group['exit_price'],
                    mode='markers',
                    name=f'Stop Loss ({type})',
                    marker=dict(
                        symbol='x',
                        size=12,
                        color='red',
                        line=dict(width=2, color=border_color)
                    )
                ),
                row=candlestick_row, col=1
            )

    fig_main.show()

In [ ]:
from SRC.LIBRARIES.binance_metrics import *

# ========== ЗАПУСК ТЕСТОВ ДЛЯ ВСЕХ ПЕРИОДОВ ==========
print(f'MARKET TYPE: {market_type}')
print(f'RED_MAX_BARS_OUTSIDE: {RED_MAX_BARS_OUTSIDE}')
print(f'ENABLE_RED_CHANNEL: {ENABLE_RED_CHANNEL}')
print(f'ENABLE_GREEN_CHANNEL: {ENABLE_GREEN_CHANNEL}')
print(f'RED DIRECTION CHANGE FILTER: {"ENABLED" if RED_ENABLE_DIRECTION_CHANGE_FILTER else "DISABLED"}')
print(f'RED_REQUIRE_GREEN_TOUCH_AFTER_SL: {"ENABLED" if RED_REQUIRE_GREEN_TOUCH_AFTER_SL else "DISABLED"}')
print(f'RED STOCH FILTER: {"ENABLED" if RED_STOCH_FILTER else "DISABLED"}')
print(f'RED_ENABLE_SIGNED_OI_SLOPE_FILTER: {"ENABLED" if RED_ENABLE_SIGNED_OI_SLOPE_FILTER else "DISABLED"}')
print(f'RED_SIGNED_OI_SLOPE_THRESHOLD: {RED_SIGNED_OI_SLOPE_THRESHOLD}')
print(f"RED OI regression window: {RED_OI_REGRESSION_LOOKBACK_5M * 5} minutes ({RED_OI_REGRESSION_LOOKBACK_5M} x 5m bars)")

all_periods_pnl = 0

for start_date_str, end_date_str in periods:
    print(f"\n{'='*80}")
    print(f"ПЕРИОД: {start_date_str} - {end_date_str}")

    display_start_dt = parser.parse(start_date_str)
    load_end_date = parser.parse(end_date_str)

    tf_value = int(TF[:-1])
    timeframe_seconds = {
        'M': tf_value * 60,
        'H': tf_value * 3600,
        'D': tf_value * 86400
    }.get(TF[-1], 1800)

    load_start_dt = display_start_dt - timedelta(seconds=timeframe_seconds * mrc_buffer)

    mrc_df = load_from_cache_or_fetch_df(tf=TF, cache_subdir='klines_mrc', start_dt=load_start_dt, end_dt=load_end_date)
    df = mrc_df.iloc[mrc_buffer:].copy()

    # Убеждаемся, что индексы уникальны
    mrc_df = mrc_df[~mrc_df.index.duplicated(keep='first')]
    df = df[~df.index.duplicated(keep='first')]

    # Stochastic
    stoch_calc_df = nu.prepare_buffer_data(mrc_df, df, stoch_buffer)
    df = nu.stochastic_tradingview(df, stoch_calc_df)

    # ATR
    atr_calc_df = nu.prepare_buffer_data(mrc_df, df, atr_buffer)
    df = nu.atr(df, atr_calc_df)

    # Расчёт MRC
    mrc_df = nu.mrc_calculate(mrc_df, df)

    # Сохраняем df_counter
    df_counter = mrc_df.iloc[mrc_buffer - 1 - max(RED_MIN_BARS_OUTSIDE, RED_DIRECTION_CHANGE_LOOKBACK, GREEN_MIN_BARS_OUTSIDE):].copy()

    # ATR df_counter
    atr_calc_df_counter = nu.prepare_buffer_data(mrc_df, df_counter, atr_buffer)
    df_counter = nu.atr(df_counter, atr_calc_df_counter)
    df_counter['meanline_direction_changes'] = calculate_meanline_direction_changes(df_counter, RED_DIRECTION_CHANGE_LOOKBACK)

    binance_metrics_df = load_binance_metrics(symbol=SYMBOL, start_date=df_counter.index.min(), end_date=df_counter.index.max())

    binance_metrics_df["sum_open_interest_pct_change"] = binance_metrics_df["sum_open_interest"].pct_change()
    # TODO RED_OI_REGRESSION_LOOKBACK_5M is used here. Change this for the Green channel.
    binance_metrics_df = add_regression_metrics(
        df=binance_metrics_df,
        source_column="sum_open_interest_pct_change",
        lookback=RED_OI_REGRESSION_LOOKBACK_5M,
    )

    df_counter = attach_binance_metrics(tf=TF, df_counter=df_counter, metrics_df=binance_metrics_df)
    df_1m = load_from_cache_or_fetch_df(tf='1M', cache_subdir='klines_1M', start_dt=display_start_dt, end_dt=load_end_date)

    all_trades = []

    find_break_candle_idx_offset = len(df_counter) - len(df)

    if ENABLE_RED_CHANNEL:
        red_trades = backtest_red_channel(
            df=df,
            df_counter=df_counter,
            df_1m = df_1m,
            commission=COMMISSION_RATE,
            capital_per_trade=CAPITAL_PER_TRADE,
            slippage=SLIPPAGE,
            tp_level=RED_TP_LEVEL,
            tp_mode=RED_TP_MODE,
            sl_ratio=RED_SL_RATIO,
            require_green_touch_after_sl=RED_REQUIRE_GREEN_TOUCH_AFTER_SL,
            min_bars_outside=RED_MIN_BARS_OUTSIDE,
            max_bars_outside=RED_MAX_BARS_OUTSIDE,
            stoch_filter=RED_STOCH_FILTER,
            stoch_series=RED_STOCH_SERIES,
            stoch_oversold=RED_STOCH_OVERSOLD,
            stoch_overbought=RED_STOCH_OVERBOUGHT,
            enable_signed_oi_slope_filter=RED_ENABLE_SIGNED_OI_SLOPE_FILTER,
            signed_oi_slope_threshold=RED_SIGNED_OI_SLOPE_THRESHOLD,
            force_close_at_end=FORCE_CLOSE_AT_END,
            min_break_body_to_atr=RED_MIN_BREAK_BODY_TO_ATR,
            find_break_candle_idx_offset=find_break_candle_idx_offset,
            enable_direction_change_filter=RED_ENABLE_DIRECTION_CHANGE_FILTER
        )
        all_trades.extend(red_trades)

    if ENABLE_GREEN_CHANNEL:
        green_trades = backtest_green_channel(
            df=df,
            df_counter=df_counter,
            commission=COMMISSION_RATE,
            capital_per_trade=CAPITAL_PER_TRADE,
            slippage=SLIPPAGE,
            tp_level=GREEN_TP_LEVEL,
            tp_mode=GREEN_TP_MODE,
            require_yellow_touch_after_sl=GREEN_REQUIRE_YELLOW_TOUCH_AFTER_SL,
            min_bars_outside=GREEN_MIN_BARS_OUTSIDE,
            max_bars_outside=GREEN_MAX_BARS_OUTSIDE,
            stoch_filter=GREEN_STOCH_FILTER,
            stoch_series=GREEN_STOCH_SERIES,
            stoch_oversold=GREEN_STOCH_OVERSOLD,
            stoch_overbought=GREEN_STOCH_OVERBOUGHT,
            force_close_at_end=FORCE_CLOSE_AT_END,
            min_break_body_to_atr=GREEN_MIN_BREAK_BODY_TO_ATR,
            find_break_candle_idx_offset=find_break_candle_idx_offset
        )
        all_trades.extend(green_trades)

    metrics = calculate_metrics(all_trades)
    total_pnl = metrics['total_pnl']
    all_periods_pnl += total_pnl
    print(f"Total pnL = ${total_pnl:.2f}, Trades = {metrics['num_trades']}, WR = {metrics['win_rate']:.2f}, PF = {metrics['profit_factor']:.2f}")

print(f'\nAll periods P&L ${all_periods_pnl:.2f}')
print("\n" + "="*100)

# show_fig_main()

os.system('afplay /System/Library/Sounds/Glass.aiff')